# YouTube GraphRAG Embedding Pipeline (production)

This notebook:

- Embeds `YouTubeVideo.comment_summary_description` → `comment_summary_embedding`
- Embeds combined video text → `video_content_embedding`
- **Syncs comment topics from MongoDB** (`youtube_channel_videos`) into Neo4j as `(:YouTubeCommentTopic)` and `[:HAS_COMMENT_TOPIC]`, **only** when `(:YouTubeVideo {video_id})` already exists with the **same `video_id` string** as Mongo.
- Embeds `YouTubeCommentTopic.name` → `embedding` and ensures a dedicated vector index.

Uses Neo4j / Mongo / embedding credentials from `.env`. Re-runs are idempotent.
> **Topic pipeline (run last):** After comment-summary and video-content embedding cells, run **Comment topics — Mongo → Neo4j** then **Sync topics from Mongo…** (immediately before `driver.close()`).


In [1]:
import os
import time
from typing import Any, Dict, Iterable, Iterator, List, Optional

from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import AzureOpenAI, OpenAI
from pymongo import MongoClient

load_dotenv()
print('Env loaded')



Env loaded


In [2]:
def _redact_bolt_uri(uri: str) -> str:
    if not uri:
        return ''
    if '@' not in uri:
        return uri
    left, right = uri.rsplit('@', 1)
    if '://' in left:
        scheme = left.split('://', 1)[0]
        return f'{scheme}://***:***@{right}'
    return uri


def _redact_mongo_uri(uri: str) -> str:
    if not uri or '://' not in uri:
        return uri
    scheme, rest = uri.split('://', 1)
    if '@' not in rest:
        return uri
    _creds, hostpart = rest.rsplit('@', 1)
    return f'{scheme}://***:***@{hostpart}'


# Production Neo4j from .env
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USER = os.getenv('NEO4J_USER')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

# Mongo (same collection as DAG / youtube_embedding_test)
MONGO_URI = os.getenv('MONGODB_URI') or os.getenv('MONGO_URI')
MONGO_DB = os.getenv('MONGODB_DB') or os.getenv('MONGO_DB', 'rbl')
MONGO_COLLECTION = os.getenv('MONGO_COLLECTION', 'youtube_channel_videos')

# Embeddings config from .env
AZURE_OPENAI_EMBEDDING_ENDPOINT = os.getenv('AZURE_OPENAI_EMBEDDING_ENDPOINT')
AZURE_OPENAI_EMBEDDING_API_KEY = os.getenv('AZURE_OPENAI_EMBEDDING_API_KEY')
AZURE_OPENAI_EMBEDDING_API_VERSION = os.getenv('AZURE_OPENAI_EMBEDDING_API_VERSION', '2024-02-01')
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = (
    os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT_MODEL_NAME')
    or os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME')
    or os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')
    or 'text-embedding-3-large'
)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-large')

# Tuning knobs
PAGE_SIZE = int(os.getenv('YT_SUMMARY_EMBED_PAGE_SIZE', '500'))
EMBED_BATCH = int(os.getenv('YT_SUMMARY_EMBED_BATCH', '128'))
WRITE_BATCH = int(os.getenv('YT_SUMMARY_WRITE_BATCH', '500'))
TOPIC_UPSERT_BATCH = int(os.getenv('YT_TOPIC_UPSERT_BATCH', '200'))
TOPIC_SYNC_LIMIT = int(os.getenv('YT_TOPIC_SYNC_LIMIT', '0'))  # 0 = scan full collection
TOPIC_EMBED_WRITE_BATCH = int(os.getenv('YT_TOPIC_EMBED_WRITE_BATCH', '32'))  # small txs for topic vector writes (Aura / prod)

YOUTUBE_PLATFORM = 'youtube'

assert NEO4J_URI and NEO4J_USER and NEO4J_PASSWORD, 'Set NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD in .env'
assert MONGO_URI, 'Set MONGODB_URI or MONGO_URI in .env for topic sync'

print('Config:')
print(f'  neo4j: {_redact_bolt_uri(NEO4J_URI)}  db={NEO4J_DATABASE}  user={NEO4J_USER}')
print(f'  mongo: {_redact_mongo_uri(MONGO_URI)}  db={MONGO_DB}  collection={MONGO_COLLECTION}')
print(f'  batches: page={PAGE_SIZE} embed={EMBED_BATCH} write={WRITE_BATCH} topic_upsert={TOPIC_UPSERT_BATCH}  topic_sync_limit={TOPIC_SYNC_LIMIT or "none"}  topic_vec_write={TOPIC_EMBED_WRITE_BATCH}')



Config:
  neo4j: bolt+ssc://rbl-neo4j.ecda.ai:7687  db=neo4j  user=neo4j
  mongo: mongodb://***:***@  db=rbl  collection=youtube_channel_videos
  batches: page=500 embed=128 write=500 topic_upsert=200  topic_sync_limit=none  topic_vec_write=32


In [3]:
def build_embedding_client():
    if AZURE_OPENAI_EMBEDDING_ENDPOINT and AZURE_OPENAI_EMBEDDING_API_KEY:
        client = AzureOpenAI(
            azure_endpoint=AZURE_OPENAI_EMBEDDING_ENDPOINT,
            api_key=AZURE_OPENAI_EMBEDDING_API_KEY,
            api_version=AZURE_OPENAI_EMBEDDING_API_VERSION,
        )
        return 'azure', client, AZURE_OPENAI_EMBEDDING_DEPLOYMENT
    if OPENAI_API_KEY:
        return 'openai', OpenAI(api_key=OPENAI_API_KEY), OPENAI_EMBEDDING_MODEL
    raise RuntimeError('No embedding credentials found in .env')


def batched(iterable: Iterable, n: int) -> Iterator[list]:
    bucket = []
    for item in iterable:
        bucket.append(item)
        if len(bucket) >= n:
            yield bucket
            bucket = []
    if bucket:
        yield bucket


EMBED_PROVIDER, embed_client, EMBED_MODEL_NAME = build_embedding_client()
probe = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=['ping'])
EMBED_DIM = len(probe.data[0].embedding)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session(database=NEO4J_DATABASE) as s:
    print('Neo4j ok:', s.run('RETURN 1 AS ok').single()['ok'])
    pending = s.run(
        'MATCH (v:YouTubeVideo) WHERE v.comment_summary_description IS NOT NULL AND v.comment_summary_embedding IS NULL RETURN count(v) AS c'
    ).single()['c']
    print('Pending YouTube summary embeddings:', pending)

mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=30_000)
mongo_coll = mongo_client[MONGO_DB][MONGO_COLLECTION]
mongo_client.admin.command('ping')
print('Mongo ok:', MONGO_DB, MONGO_COLLECTION)

print(f'Embedding provider: {EMBED_PROVIDER}  model/deployment: {EMBED_MODEL_NAME}  dim={EMBED_DIM}')



Neo4j ok: 1
Pending YouTube summary embeddings: 0


/tmp/ipykernel_30152/1014088276.py:37: UserWarning: You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
  mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=30_000)


Mongo ok: rbl youtube_channel_videos
Embedding provider: azure  model/deployment: text-embedding-3-large  dim=3072


## Video comment summary embedding functions

Defines summary fetch/write Cypher and the `embed_youtube_comment_summaries` runner.

In [4]:
def ensure_vector_indexes(driver, dim: int) -> None:
    summary_stmt = f'''CREATE VECTOR INDEX video_summary_embedding_index IF NOT EXISTS
    FOR (v:YouTubeVideo) ON (v.comment_summary_embedding)
    OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}'''
    content_stmt = f'''CREATE VECTOR INDEX video_content_embedding_index IF NOT EXISTS
    FOR (v:YouTubeVideo) ON (v.video_content_embedding)
    OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}'''
    with driver.session(database=NEO4J_DATABASE) as s:
        s.run(summary_stmt)
        s.run(content_stmt)
    print('Ensured video_summary_embedding_index and video_content_embedding_index')


def build_video_content_text(
    title: Optional[str],
    description: Optional[str],
    thumbnail_description: Optional[str],
    thumbnail_keywords: List[str],
    tags: List[str],
) -> Optional[str]:
    parts: List[str] = []
    if title:
        parts.append(f'Title: {title}')
    if description:
        parts.append(f'Description: {description}')
    if thumbnail_description:
        parts.append(f'Thumbnail: {thumbnail_description}')
    if thumbnail_keywords:
        parts.append(f'Thumbnail keywords: {", ".join(thumbnail_keywords)}')
    if tags:
        parts.append(f'Tags: {", ".join(tags)}')
    return '\n'.join(parts).strip() if parts else None


def embed_texts(texts: List[str], max_retries: int = 5) -> List[List[float]]:
    out: List[List[float]] = []
    for chunk in batched(texts, EMBED_BATCH):
        attempt = 0
        while True:
            try:
                resp = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=chunk)
                out.extend([d.embedding for d in resp.data])
                break
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise
                wait = min(2 ** attempt, 30)
                print(f'  embed retry {attempt} after {wait}s: {e}')
                time.sleep(wait)
    return out


FETCH_SUMMARY_CYPHER = '''
MATCH (v:YouTubeVideo)
WHERE v.comment_summary_description IS NOT NULL
  AND v.comment_summary_embedding IS NULL
RETURN elementId(v) AS eid, v.comment_summary_description AS text
LIMIT $limit
'''

WRITE_SUMMARY_CYPHER = '''
UNWIND $rows AS row
MATCH (v) WHERE elementId(v) = row.eid
CALL db.create.setNodeVectorProperty(v, 'comment_summary_embedding', row.embedding)
RETURN count(*) AS written
'''

def embed_youtube_comment_summaries(driver, page_size: int = PAGE_SIZE) -> int:
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(s.run(FETCH_SUMMARY_CYPHER, limit=page_size))
        if not pending:
            break

        texts = [r['text'] for r in pending]
        embeddings = embed_texts(texts)
        rows = [{'eid': pending[i]['eid'], 'embedding': embeddings[i]} for i in range(len(pending))]

        for chunk in batched(rows, WRITE_BATCH):
            with driver.session(database=NEO4J_DATABASE) as s:
                written = s.run(WRITE_SUMMARY_CYPHER, rows=chunk).single()['written']
            total_written += written

        print(f'  summary written={total_written}  elapsed={time.time()-start:.1f}s')

    return total_written

In [ ]:
# Summary status check (under comment summary section)
with driver.session(database=NEO4J_DATABASE) as s:
    summary_totals = {
        'videos_with_summary': s.run('MATCH (v:YouTubeVideo) WHERE v.comment_summary_description IS NOT NULL RETURN count(v) AS c').single()['c'],
        'videos_with_summary_embedding': s.run('MATCH (v:YouTubeVideo) WHERE v.comment_summary_embedding IS NOT NULL RETURN count(v) AS c').single()['c'],
        'summary_pending': s.run('MATCH (v:YouTubeVideo) WHERE v.comment_summary_description IS NOT NULL AND v.comment_summary_embedding IS NULL RETURN count(v) AS c').single()['c'],
    }

print(summary_totals)

## Video content embedding functions

Defines helper text construction, content fetch/write Cypher, and the `embed_youtube_video_content` runner.

In [ ]:
# Video content embedding helpers and cypher
from typing import List, Optional

def build_video_content_text(
    title: Optional[str],
    description: Optional[str],
    thumbnail_description: Optional[str],
    thumbnail_keywords: List[str],
    tags: List[str],
) -> Optional[str]:
    parts: List[str] = []
    if title:
        parts.append(f'Title: {title}')
    if description:
        parts.append(f'Description: {description}')
    if thumbnail_description:
        parts.append(f'Thumbnail: {thumbnail_description}')
    if thumbnail_keywords:
        parts.append(f'Thumbnail keywords: {", ".join(thumbnail_keywords)}')
    if tags:
        parts.append(f'Tags: {", ".join(tags)}')
    return '\n'.join(parts).strip() if parts else None


FETCH_CONTENT_CYPHER = '''
MATCH (v:YouTubeVideo)
WHERE v.video_content_embedding IS NULL
  AND (
    v.video_title IS NOT NULL
    OR v.video_description IS NOT NULL
    OR v.thumbnail_description IS NOT NULL
    OR size(coalesce(v.thumbnail_keywords, [])) > 0
    OR size(coalesce(v.tags, [])) > 0
  )
RETURN elementId(v) AS eid,
       v.video_title AS video_title,
       v.video_description AS video_description,
       v.thumbnail_description AS thumbnail_description,
       coalesce(v.thumbnail_keywords, []) AS thumbnail_keywords,
       coalesce(v.tags, []) AS tags
LIMIT $limit
'''

WRITE_CONTENT_CYPHER = '''
UNWIND $rows AS row
MATCH (v) WHERE elementId(v) = row.eid
SET v.video_content_text = row.text
WITH v, row
CALL db.create.setNodeVectorProperty(v, 'video_content_embedding', row.embedding)
RETURN count(*) AS written
'''


def embed_youtube_video_content(driver, page_size: int = PAGE_SIZE) -> int:
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(s.run(FETCH_CONTENT_CYPHER, limit=page_size))
        if not pending:
            break

        rows_to_embed = []
        texts = []
        for record in pending:
            thumbnail_keywords = [k for k in (record['thumbnail_keywords'] or []) if isinstance(k, str)]
            tags = [t for t in (record['tags'] or []) if isinstance(t, str)]
            text = build_video_content_text(
                record['video_title'],
                record['video_description'],
                record['thumbnail_description'],
                thumbnail_keywords,
                tags,
            )
            if text:
                rows_to_embed.append({'eid': record['eid'], 'text': text})
                texts.append(text)

        if not rows_to_embed:
            break

        embeddings = embed_texts(texts)
        write_rows = [
            {'eid': rows_to_embed[i]['eid'], 'text': rows_to_embed[i]['text'], 'embedding': embeddings[i]}
            for i in range(len(rows_to_embed))
        ]

        for chunk in batched(write_rows, WRITE_BATCH):
            with driver.session(database=NEO4J_DATABASE) as s:
                written = s.run(WRITE_CONTENT_CYPHER, rows=chunk).single()['written']
            total_written += written

        print(f'  content written={total_written}  elapsed={time.time()-start:.1f}s')

    return total_written

## Run video content embedding

Runs only `video_content_embedding` creation and reports content embedding totals.

In [ ]:
# Run video content embeddings only (comment summary already completed)
with driver.session(database=NEO4J_DATABASE) as s:
    s.run(
        f'''CREATE VECTOR INDEX video_content_embedding_index IF NOT EXISTS
        FOR (v:YouTubeVideo) ON (v.video_content_embedding)
        OPTIONS {{ indexConfig: {{ `vector.dimensions`: {EMBED_DIM}, `vector.similarity_function`: 'cosine' }} }}'''
    )

content_written = embed_youtube_video_content(driver)
print('Done. Newly embedded video content:', content_written)

with driver.session(database=NEO4J_DATABASE) as s:
    content_totals = {
        'videos_with_content_embedding': s.run('MATCH (v:YouTubeVideo) WHERE v.video_content_embedding IS NOT NULL RETURN count(v) AS c').single()['c'],
        'content_pending': s.run("MATCH (v:YouTubeVideo) WHERE v.video_content_embedding IS NULL AND (v.video_title IS NOT NULL OR v.video_description IS NOT NULL OR v.thumbnail_description IS NOT NULL OR size(coalesce(v.thumbnail_keywords, [])) > 0 OR size(coalesce(v.tags, [])) > 0) RETURN count(v) AS c").single()['c'],
    }

print(content_totals)

## Comment topics — Mongo → Neo4j (`YouTubeCommentTopic`)

Reads `comments_frequent_topics` (+ categories / weights) from `youtube_channel_videos`. For each document, **only** if `(:YouTubeVideo {video_id})` exists in Neo4j with the same **string** `video_id` as Mongo, existing `HAS_COMMENT_TOPIC` relationships from that video are removed and recreated pointing at `(:YouTubeCommentTopic {video_id, name})`.

Videos that exist in Mongo but not yet in Neo4j are **skipped** (no orphan topics). Empty topic lists still clear stale relationships on matched videos.
**Mongo field fidelity (nothing shortened in code):**
- Topic sync uses a projection only to choose **which fields** to load (`video_id`, topic names/categories/weights). Values are read **in full** — arrays are not sliced and strings are not truncated.
- Every entry in `comments_frequent_topics` is processed (no max-topics cap). We skip only rows whose **name** is empty after optional coercion.
- `clean_text` does **not** shorten strings; it only converts non-strings with `str()` for Neo4j-safe storage.
- `video_id` is normalized to string and **stripped of leading/trailing whitespace** so it matches Neo4j — we do not trim the inner content.
- `TOPIC_SYNC_LIMIT` / `TT_TOPIC_SYNC_LIMIT` only limits how many **documents** are scanned when set non-zero; it does not clip fields inside a document.

Embedding APIs (Azure/OpenAI) may enforce their **own** token limits per request — that is outside this notebook.


In [ ]:
TOPICS_MONGO_PROJECTION = {
    'video_id': 1,
    'comments_frequent_topics': 1,
    'comments_frequent_topic_categories': 1,
    'comments_frequent_topic_weights': 1,
}


def clean_text(value: Any) -> Optional[str]:
    """Normalize scalars for Neo4j; never truncate string length."""
    if value is None:
        return None
    if isinstance(value, str):
        return value
    return str(value)


def topic_row_from_mongo_doc(doc: dict) -> Optional[Dict[str, Any]]:
    """One Mongo doc → {video_id: str, topics: [...]}.
    Processes the full `comments_frequent_topics` list (no slicing/cap). Categories/weights aligned by index."""
    raw_id = doc.get('video_id')
    if raw_id is None:
        return None
    video_id = str(raw_id).strip()
    if not video_id:
        return None
    names = doc.get('comments_frequent_topics') or []
    categories = doc.get('comments_frequent_topic_categories') or []
    weights = doc.get('comments_frequent_topic_weights') or []
    topics: List[Dict[str, Any]] = []
    for i, name in enumerate(names):
        name = clean_text(name)
        if not name:
            continue
        topics.append(
            {
                'name': name,
                'category': clean_text(categories[i] if i < len(categories) else None),
                'weight': float(weights[i]) if i < len(weights) and weights[i] is not None else None,
                'position': i,
            }
        )
    return {'video_id': video_id, 'topics': topics}


SYNC_YOUTUBE_COMMENT_TOPICS_CYPHER = """
UNWIND $rows AS row
OPTIONAL MATCH (v:YouTubeVideo {video_id: row.video_id})
WITH row, v
WHERE v IS NOT NULL
OPTIONAL MATCH (v)-[r:HAS_COMMENT_TOPIC]->()
DELETE r
WITH row, v
UNWIND row.topics AS topic
WITH row, v, topic
WHERE topic.name IS NOT NULL
MERGE (ct:YouTubeCommentTopic {video_id: row.video_id, name: topic.name})
SET ct.category = topic.category,
    ct.platform = $youtube_platform
MERGE (v)-[rel:HAS_COMMENT_TOPIC]->(ct)
SET rel.weight = topic.weight,
    rel.position = topic.position,
    rel.platform = $youtube_platform
"""


def ensure_youtube_comment_topic_constraint(driver) -> None:
    stmt = (
        'CREATE CONSTRAINT youtube_comment_topic_video_name IF NOT EXISTS '
        'FOR (t:YouTubeCommentTopic) REQUIRE (t.video_id, t.name) IS UNIQUE'
    )
    with driver.session(database=NEO4J_DATABASE) as s:
        s.run(stmt)
    print('Ensured constraint on (YouTubeCommentTopic.video_id, YouTubeCommentTopic.name)')


def ensure_youtube_comment_topic_vector_index(driver, dim: int) -> None:
    stmt = f"""CREATE VECTOR INDEX youtube_comment_topic_embedding_index IF NOT EXISTS
    FOR (t:YouTubeCommentTopic) ON (t.embedding)
    OPTIONS {{ indexConfig: {{ `vector.dimensions`: {dim}, `vector.similarity_function`: 'cosine' }} }}"""
    with driver.session(database=NEO4J_DATABASE) as s:
        s.run(stmt)
    print('Ensured youtube_comment_topic_embedding_index')


def sync_youtube_comment_topics_from_mongo(
    driver,
    collection,
    batch_size: int = TOPIC_UPSERT_BATCH,
) -> tuple[int, int]:
    """Returns (mongo_docs_in_batches, topic_slot_rows_synced) — count of topic entries processed (≤10×videos when capped)."""
    cur = collection.find({}, TOPICS_MONGO_PROJECTION)
    if TOPIC_SYNC_LIMIT > 0:
        cur = cur.limit(TOPIC_SYNC_LIMIT)
    total_docs = 0
    total_links = 0
    for batch in batched(cur, batch_size):
        rows = []
        for doc in batch:
            row = topic_row_from_mongo_doc(doc)
            if row:
                rows.append(row)
        if not rows:
            continue
        total_docs += len(rows)
        batch_topic_slots = sum(len(r['topics']) for r in rows)
        with driver.session(database=NEO4J_DATABASE) as s:
            s.run(
                SYNC_YOUTUBE_COMMENT_TOPICS_CYPHER,
                rows=rows,
                youtube_platform=YOUTUBE_PLATFORM,
            )
        total_links += batch_topic_slots
        print(f'  topic sync: mongo_rows={total_docs}  topic_slots_synced={total_links}')
    return total_docs, total_links


YOUTUBE_COMMENT_TOPIC_FETCH_CYPHER = """
MATCH (t:YouTubeCommentTopic)
WHERE coalesce(t.platform, 'youtube') = $platform
  AND t.embedding IS NULL AND t.name IS NOT NULL
RETURN elementId(t) AS eid, t.name AS name
LIMIT $limit
"""

YOUTUBE_COMMENT_TOPIC_WRITE_CYPHER = """
UNWIND $rows AS row
MATCH (t) WHERE elementId(t) = row.eid
CALL db.create.setNodeVectorProperty(t, 'embedding', row.embedding)
RETURN count(*) AS written
"""


def _embed_texts_for_topics(texts: List[str], max_retries: int = 5) -> List[List[float]]:
    """Same logic as `embed_texts` in the summary section — uses embed_client from setup so topic embedding works without running summary cells."""
    out: List[List[float]] = []
    for chunk in batched(texts, EMBED_BATCH):
        attempt = 0
        while True:
            try:
                resp = embed_client.embeddings.create(model=EMBED_MODEL_NAME, input=chunk)
                out.extend([d.embedding for d in resp.data])
                break
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise
                wait = min(2 ** attempt, 30)
                print(f'  embed retry {attempt} after {wait}s: {e}')
                time.sleep(wait)
    return out


def _write_topic_embedding_chunk(driver, rows_chunk):
    """Write topic vectors in small Neo4j transactions; retry on transient/store errors (Aura)."""
    from neo4j.exceptions import Neo4jError

    attempt = 0
    while True:
        try:
            with driver.session(database=NEO4J_DATABASE) as s:
                rec = s.run(YOUTUBE_COMMENT_TOPIC_WRITE_CYPHER, rows=rows_chunk).single()
                return int(rec['written']) if rec else 0
        except Neo4jError as e:
            attempt += 1
            if attempt > 8:
                raise
            wait = min(2 ** attempt, 120)
            print(f'  neo4j vector write retry {attempt} after {wait}s: {e}')
            time.sleep(wait)

def embed_youtube_comment_topic_nodes(driver, page_size: int = PAGE_SIZE) -> int:
    total_written = 0
    start = time.time()
    while True:
        with driver.session(database=NEO4J_DATABASE) as s:
            pending = list(
                s.run(
                    YOUTUBE_COMMENT_TOPIC_FETCH_CYPHER,
                    limit=page_size,
                    platform=YOUTUBE_PLATFORM,
                )
            )
        if not pending:
            break
        texts = [r['name'] for r in pending]
        embeddings = _embed_texts_for_topics(texts)
        rows = [
            {'eid': pending[i]['eid'], 'embedding': embeddings[i]}
            for i in range(len(pending))
        ]
        for chunk in batched(rows, TOPIC_EMBED_WRITE_BATCH):
            written = _write_topic_embedding_chunk(driver, chunk)
            total_written += written
        print(f'  YouTubeCommentTopic embeddings written={total_written}  elapsed={time.time() - start:.1f}s')
    return total_written



In [ ]:
# Sync topics from Mongo, then embed YouTubeCommentTopic.name
# Requires: setup cell (`driver`, `mongo_coll`, `embed_client`, `EMBED_DIM`) and — if you want summary/content vectors — those sections above.

ensure_youtube_comment_topic_constraint(driver)
mongo_docs, topic_links = sync_youtube_comment_topics_from_mongo(driver, mongo_coll)
print('Topic sync done. mongo_docs (batched rows):', mongo_docs, '  topic_slots:', topic_links)

ensure_youtube_comment_topic_vector_index(driver, EMBED_DIM)
topic_embed_written = embed_youtube_comment_topic_nodes(driver)
print('YouTubeCommentTopic embedding rows written:', topic_embed_written)

with driver.session(database=NEO4J_DATABASE) as s:
    stats = {
        'YouTubeCommentTopic': s.run('MATCH (t:YouTubeCommentTopic) RETURN count(t) AS c').single()['c'],
        'HAS_COMMENT_TOPIC': s.run('MATCH ()-[r:HAS_COMMENT_TOPIC]->(:YouTubeCommentTopic) RETURN count(r) AS c').single()['c'],
        'topics_with_embedding': s.run(
            "MATCH (t:YouTubeCommentTopic) WHERE t.embedding IS NOT NULL RETURN count(t) AS c"
        ).single()['c'],
    }
print(stats)



In [ ]:
_d = globals().get('driver')
if _d is not None:
    try:
        _d.close()
    except Exception:
        pass
    print('Closed Neo4j driver')
else:
    print('No Neo4j driver in memory — run the setup cell that creates `driver` first.')
_mc = globals().get('mongo_client')
if _mc is not None:
    try:
        _mc.close()
    except Exception:
        pass
    print('Closed Mongo client')
else:
    print('No Mongo client in memory — skipped.')

## Test queries

These are the retrieval primitives the AI chat app will use. Each takes a theme (e.g. `"vaping"`, `"imagery of weapons or violence"`, `"gambling"`), embeds it, then hits one of the Neo4j vector indexes. All accept optional `k` (vector top-k recall) and `min_score` (cosine similarity floor).

- **`top_videos_by_topic_engagement(theme)`** – topic-matched videos ranked by *topic strength × log1p(comment_count)*. Best for *"most popular videos where the audience discusses X the most"*.
- **`top_creators_by_topic_engagement(theme)`** – same signal aggregated per creator. Best for *"most popular creators where audiences discuss X the most"*.
- **`top_creators(theme)`** – vector search over `Topic.embedding`, aggregate weighted contribution per `Channel`. Best for *"which influencers' audiences talk about X"* (ranked top-N).
- **`count_creators_by_topic(theme, min_score=..., k=...)`** – count distinct creators with at least one comment-topic match at or above `min_score`. Best for *"how many creators talk about X"* (comment-discussion signal).
- **`count_creators_by_content(theme, min_score=..., k=...)`**
- **`count_videos_by_topic(theme)`** – count videos with comment-topic matches for X.
- **`count_videos_by_comment_summary(theme)`** – count videos whose comment-section summary matches X.
- **`count_videos_by_content(theme)`** – count videos whose on-video content matches X.
 – same count over `video_content_embedding`. Best for *"how many creators make content about X"*.
- **`example_videos(theme)`** – vector search over `Topic.embedding`, return videos whose comment-topics match. Best for *"show me examples of X content"* when you want comment-driven evidence.
- **`comment_discussions(theme)`** – vector search over `comment_summary_embedding`. Best for *"whose comment section discusses X"*.
- **`content_videos(theme)`** – vector search over `video_content_embedding` (title + description + thumbnail description + thumbnail keywords + tags). Best for **"which videos show / are about X"**, including visual imagery questions.
- **`content_top_creators(theme)`** – aggregates `content_videos` hits by creator.
- **`unified_search(theme)`** – runs all three vector indexes and fuses results with Reciprocal Rank Fusion (RRF). Use this when you don't know which signal the user's question maps to.

In [5]:
def embed_query(text: str) -> List[float]:
    """Embed a single user query string with the same model used for graph indexes."""
    return embed_texts([text])[0]

# Returns channels ranked by the sum of weighted topic-match contributions.
TOP_CREATORS_CYPHER = '''
CALL db.index.vector.queryNodes('youtube_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'youtube') = $platform
  AND score >= $min_score
MATCH (v:YouTubeVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'youtube') = $platform
OPTIONAL MATCH (c:YouTubeChannel)-[hv:HAS_VIDEO]->(v)
WHERE coalesce(hv.platform, "YouTube") = "YouTube"
WITH
  coalesce(c.title, v.channel_title, v.username, 'unknown') AS creator,
  coalesce(c.channel_id, v.channel_id) AS channel_id,
  t, v, rel, score
WITH
  creator,
  channel_id,
  sum(score * coalesce(rel.weight, 1.0)) AS relevance,
  count(DISTINCT v) AS video_count,
  collect(DISTINCT t.name)[0..5] AS sample_topics,
  collect(DISTINCT {video_id: v.video_id, title: v.video_title, url: v.video_url})[0..8] AS sample_videos
RETURN creator, channel_id, relevance, video_count, sample_topics, sample_videos
ORDER BY relevance DESC
LIMIT $top_n
'''
#the same as top_creators but for videos
EXAMPLE_VIDEOS_CYPHER = '''
CALL db.index.vector.queryNodes('youtube_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'youtube') = $platform
  AND score >= $min_score
MATCH (v:YouTubeVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'youtube') = $platform
WITH v, t, rel, score
ORDER BY score * coalesce(rel.weight, 1.0) DESC
WITH v,
     collect({topic: t.name, weight: rel.weight, score: score})[0..3] AS matches,
     max(score * coalesce(rel.weight, 1.0)) AS best
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.channel_title, v.username) AS creator,
       v.video_url AS url,
       v.view_count AS views,
       best AS relevance,
       matches
ORDER BY best DESC
LIMIT $top_n
'''
#returns top n videos with the highest similarity score from comment_summary_embedding
COMMENT_DISCUSSIONS_CYPHER = '''
CALL db.index.vector.queryNodes('video_summary_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.channel_title, v.username) AS creator,
       coalesce(v.channel_id, '') AS channel_id,
       v.video_url AS url,
       score,
       v.comment_summary_description AS comment_summary_description
ORDER BY score DESC
LIMIT $top_n
'''
#returns top n videos with the highest similarity score from video_content_embedding
CONTENT_VIDEOS_CYPHER = '''
CALL db.index.vector.queryNodes('video_content_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.channel_title, v.username) AS creator,
       coalesce(v.channel_id, '') AS channel_id,
       v.video_url AS url,
       v.view_count AS views,
       v.thumbnail_url AS thumbnail_url,
       v.thumbnail_description AS thumbnail_description,
       v.video_description AS video_description,
       v.thumbnail_keywords AS thumbnail_keywords,
       v.tags AS tags,
       score
ORDER BY score DESC
LIMIT $top_n
'''
# Returns channels ranked by the  sum of video_content_embedding similarity scores.
CONTENT_TOP_CREATORS_CYPHER = '''
CALL db.index.vector.queryNodes('video_content_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
OPTIONAL MATCH (c:YouTubeChannel)-[hv:HAS_VIDEO]->(v)
WHERE coalesce(hv.platform, "YouTube") = "YouTube"
WITH coalesce(c.title, v.channel_title, v.username, 'unknown') AS creator,
     coalesce(c.channel_id, v.channel_id) AS channel_id,
     v, score
WITH creator, channel_id,
     sum(score) AS relevance,
     count(DISTINCT v) AS video_count,
     collect({video_id: v.video_id, title: v.video_title, url: v.video_url, score: score})[0..5] AS sample_videos
RETURN creator, channel_id, relevance, video_count, sample_videos
ORDER BY relevance DESC
LIMIT $top_n
'''

# Count distinct creators with topic matches at or above min_score (bounded by vector top-k).
COUNT_CREATORS_BY_TOPIC_CYPHER = '''
CALL db.index.vector.queryNodes('youtube_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'youtube') = $platform
  AND score >= $min_score
MATCH (v:YouTubeVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'youtube') = $platform
OPTIONAL MATCH (c:YouTubeChannel)-[hv:HAS_VIDEO]->(v)
WHERE coalesce(hv.platform, "YouTube") = "YouTube"
WITH
  coalesce(c.title, v.channel_title, v.username, 'unknown') AS creator,
  coalesce(c.channel_id, v.channel_id) AS channel_id,
  v,
  score * coalesce(rel.weight, 1.0) AS weighted_score
WITH creator, channel_id,
     count(DISTINCT v) AS video_count,
     max(weighted_score) AS best_weighted_score
RETURN count(*) AS creator_count,
       sum(video_count) AS video_link_rows,
       collect({creator: creator, channel_id: channel_id, video_count: video_count, best_weighted_score: best_weighted_score})[0..20] AS sample_creators
'''

# Count distinct creators with video-content matches at or above min_score.
COUNT_CREATORS_BY_CONTENT_CYPHER = '''
CALL db.index.vector.queryNodes('video_content_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
OPTIONAL MATCH (c:YouTubeChannel)-[hv:HAS_VIDEO]->(v)
WHERE coalesce(hv.platform, "YouTube") = "YouTube"
WITH
  coalesce(c.title, v.channel_title, v.username, 'unknown') AS creator,
  coalesce(c.channel_id, v.channel_id) AS channel_id,
  v,
  score
WITH creator, channel_id,
     count(DISTINCT v) AS video_count,
     max(score) AS best_score
RETURN count(*) AS creator_count,
       sum(video_count) AS video_link_rows,
       collect({creator: creator, channel_id: channel_id, video_count: video_count, best_score: best_score})[0..20] AS sample_creators
'''

# Count distinct videos with comment-topic matches at or above min_score.
COUNT_VIDEOS_BY_TOPIC_CYPHER = '''
CALL db.index.vector.queryNodes('youtube_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'youtube') = $platform
  AND score >= $min_score
MATCH (v:YouTubeVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'youtube') = $platform
WITH v, score * coalesce(rel.weight, 1.0) AS weighted_score
WITH v, max(weighted_score) AS best_weighted_score
RETURN count(v) AS video_count,
       collect({
         video_id: v.video_id,
         title: v.video_title,
         creator: coalesce(v.channel_title, v.username),
         url: v.video_url,
         best_weighted_score: best_weighted_score
       })[0..20] AS sample_videos
'''

# Count distinct videos whose comment-summary embedding matches at or above min_score.
COUNT_VIDEOS_BY_COMMENT_SUMMARY_CYPHER = '''
CALL db.index.vector.queryNodes('video_summary_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
WITH v, max(score) AS best_score
RETURN count(v) AS video_count,
       collect({
         video_id: v.video_id,
         title: v.video_title,
         creator: coalesce(v.channel_title, v.username),
         url: v.video_url,
         score: best_score
       })[0..20] AS sample_videos
'''

# Count distinct videos with video-content matches at or above min_score.
COUNT_VIDEOS_BY_CONTENT_CYPHER = '''
CALL db.index.vector.queryNodes('video_content_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
WITH v, max(score) AS best_score
RETURN count(v) AS video_count,
       collect({
         video_id: v.video_id,
         title: v.video_title,
         creator: coalesce(v.channel_title, v.username),
         url: v.video_url,
         score: best_score
       })[0..20] AS sample_videos
'''
# Rank videos by topic match strength × log1p(comment_count).
TOP_VIDEOS_BY_TOPIC_ENGAGEMENT_CYPHER = '''
CALL db.index.vector.queryNodes('youtube_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'youtube') = $platform
  AND score >= $min_score
MATCH (v:YouTubeVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'youtube') = $platform
WITH v, t, rel, score,
  score * coalesce(rel.weight, 1.0) AS topic_strength,
  log(1.0 + toFloat(coalesce(v.comment_count, 0))) AS log_comments
WITH v,
  max(topic_strength) AS best_topic_strength,
  max(log_comments) AS log_comments,
  collect(DISTINCT {topic: t.name, weight: rel.weight, score: score})[0..3] AS matches
WITH v, best_topic_strength, log_comments, matches,
  best_topic_strength * log_comments AS engagement_score
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(v.channel_title, v.username) AS creator,
       coalesce(v.channel_id, '') AS channel_id,
       v.video_url AS url,
       toInteger(coalesce(v.comment_count, 0)) AS comment_count,
       round(best_topic_strength, 5) AS best_topic_strength,
       round(log_comments, 5) AS log_comments,
       round(engagement_score, 5) AS engagement_score,
       matches
ORDER BY engagement_score DESC
LIMIT $top_n
'''

# Rank creators by summed per-video engagement (topic strength × log comments).
TOP_CREATORS_BY_TOPIC_ENGAGEMENT_CYPHER = '''
CALL db.index.vector.queryNodes('youtube_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'youtube') = $platform
  AND score >= $min_score
MATCH (v:YouTubeVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'youtube') = $platform
OPTIONAL MATCH (c:YouTubeChannel)-[hv:HAS_VIDEO]->(v)
WHERE coalesce(hv.platform, "YouTube") = "YouTube"
WITH
  coalesce(c.title, v.channel_title, v.username, 'unknown') AS creator,
  coalesce(c.channel_id, v.channel_id) AS channel_id,
  v, t, rel, score,
  score * coalesce(rel.weight, 1.0) AS topic_strength,
  log(1.0 + toFloat(coalesce(v.comment_count, 0))) AS log_comments
WITH creator, channel_id, v,
  max(topic_strength) AS video_topic_strength,
  max(log_comments) AS log_comments,
  collect(DISTINCT t.name)[0..5] AS video_topics
WITH creator, channel_id, v, video_topics,
  video_topic_strength * log_comments AS video_engagement,
  toInteger(coalesce(v.comment_count, 0)) AS comment_count
WITH creator, channel_id,
  sum(video_engagement) AS engagement_score,
  count(DISTINCT v) AS video_count,
  sum(comment_count) AS total_comment_count,
  collect(DISTINCT video_topics)[0..5] AS sample_topics,
  collect({
    video_id: v.video_id,
    title: v.video_title,
    url: v.video_url,
    comment_count: comment_count,
    engagement_score: round(video_engagement, 5)
  })[0..8] AS sample_videos
RETURN creator,
       channel_id,
       round(engagement_score, 5) AS engagement_score,
       video_count,
       total_comment_count,
       sample_topics,
       sample_videos
ORDER BY engagement_score DESC
LIMIT $top_n
'''



def top_creators(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Rank creators whose audiences discuss `theme` (comment-topic signal).

    Index: `youtube_comment_topic_embedding_index` on `YouTubeCommentTopic`.
    Ranks by: sum(vector_score × rel.weight) per channel.
    Best for: *"which creators have audiences discussing X?"* (quality / relevance, top-N).
    Returns: creator, channel_id, relevance, video_count, sample_topics, sample_videos.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            TOP_CREATORS_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
            platform=YOUTUBE_PLATFORM,
        )]


def count_creators_by_topic(
    theme: str,
    min_score: float = 0.0,
    k: int = 2000,
) -> Dict[str, Any]:
    """Count distinct creators with comment-topic matches for `theme` at or above `min_score`.

    Index: `youtube_comment_topic_embedding_index` (same joins as `top_creators`).
    Best for: *"how many creators talk about X in comments?"* (scale, not a ranked list).
    Returns: creator_count, video_link_rows, sample_creators; bounded by vector top-k.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        rec = s.run(
            COUNT_CREATORS_BY_TOPIC_CYPHER,
            q=q,
            k=k,
            min_score=min_score,
            platform=YOUTUBE_PLATFORM,
        ).single()
    return {
        'theme': theme,
        'signal': 'comment_topics',
        'min_score': min_score,
        'k': k,
        'creator_count': int(rec['creator_count']) if rec else 0,
        'video_link_rows': int(rec['video_link_rows']) if rec else 0,
        'sample_creators': list(rec['sample_creators'] or []),
    }


def count_creators_by_content(
    theme: str,
    min_score: float = 0.0,
    k: int = 2000,
) -> Dict[str, Any]:
    """Count distinct creators with video-content matches for `theme` at or above `min_score`.

    Index: `video_content_embedding_index` on `YouTubeVideo` (title, description, thumbnail, tags).
    Best for: *"how many creators make content about X?"* (not comment-audience signal).
    Returns: creator_count, video_link_rows, sample_creators; bounded by vector top-k.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        rec = s.run(
            COUNT_CREATORS_BY_CONTENT_CYPHER,
            q=q,
            k=k,
            min_score=min_score,
        ).single()
    return {
        'theme': theme,
        'signal': 'video_content',
        'min_score': min_score,
        'k': k,
        'creator_count': int(rec['creator_count']) if rec else 0,
        'video_link_rows': int(rec['video_link_rows']) if rec else 0,
        'sample_creators': list(rec['sample_creators'] or []),
    }


def count_videos_by_topic(
    theme: str,
    min_score: float = 0.0,
    k: int = 2000,
) -> Dict[str, Any]:
    """Count distinct videos with comment-topic matches for `theme` at or above `min_score`.

    Index: `youtube_comment_topic_embedding_index` (same path as `example_videos`).
    Best for: *"how many videos discuss X in comment topics?"*
    Returns: video_count, sample_videos; bounded by vector top-k.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        rec = s.run(
            COUNT_VIDEOS_BY_TOPIC_CYPHER,
            q=q,
            k=k,
            min_score=min_score,
            platform=YOUTUBE_PLATFORM,
        ).single()
    return {
        'theme': theme,
        'signal': 'comment_topics',
        'min_score': min_score,
        'k': k,
        'video_count': int(rec['video_count']) if rec else 0,
        'sample_videos': list(rec['sample_videos'] or []),
    }


def count_videos_by_comment_summary(
    theme: str,
    min_score: float = 0.0,
    k: int = 2000,
) -> Dict[str, Any]:
    """Count distinct videos whose comment-section summary matches `theme`.

    Index: `video_summary_embedding_index` (same path as `comment_discussions`).
    Best for: *"how many videos have comment sections that discuss X?"*
    Returns: video_count, sample_videos; bounded by vector top-k.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        rec = s.run(
            COUNT_VIDEOS_BY_COMMENT_SUMMARY_CYPHER,
            q=q,
            k=k,
            min_score=min_score,
        ).single()
    return {
        'theme': theme,
        'signal': 'comment_summary',
        'min_score': min_score,
        'k': k,
        'video_count': int(rec['video_count']) if rec else 0,
        'sample_videos': list(rec['sample_videos'] or []),
    }


def count_videos_by_content(
    theme: str,
    min_score: float = 0.0,
    k: int = 2000,
) -> Dict[str, Any]:
    """Count distinct videos whose on-video content matches `theme`.

    Index: `video_content_embedding_index` (same path as `content_videos`).
    Best for: *"how many videos are about / show X?"* (not comment signal).
    Returns: video_count, sample_videos; bounded by vector top-k.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        rec = s.run(
            COUNT_VIDEOS_BY_CONTENT_CYPHER,
            q=q,
            k=k,
            min_score=min_score,
        ).single()
    return {
        'theme': theme,
        'signal': 'video_content',
        'min_score': min_score,
        'k': k,
        'video_count': int(rec['video_count']) if rec else 0,
        'sample_videos': list(rec['sample_videos'] or []),
    }


def top_videos_by_topic_engagement(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Rank videos where audiences discuss `theme` the most at comment volume (engagement).

    Index: `youtube_comment_topic_embedding_index` + `HAS_COMMENT_TOPIC` weights.
    Ranks by: max(score × weight) per video × log1p(comment_count).
    Best for: *"most popular videos where people discuss X in comments"*.
    Returns: video_id, title, creator, comment_count, engagement_score, matches, …
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            TOP_VIDEOS_BY_TOPIC_ENGAGEMENT_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
            platform=YOUTUBE_PLATFORM,
        )]


def top_creators_by_topic_engagement(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Rank creators by summed discussion engagement for `theme` (topic match × comment volume).

    Index: same as `top_videos_by_topic_engagement`, aggregated per channel.
    Ranks by: sum over videos of max(score × weight) × log1p(comment_count).
    Best for: *"which popular creators have the biggest audience discussions about X?"*.
    Returns: engagement_score, video_count, total_comment_count, sample_topics, sample_videos.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            TOP_CREATORS_BY_TOPIC_ENGAGEMENT_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
            platform=YOUTUBE_PLATFORM,
        )]



def example_videos(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Return example videos with the strongest comment-topic matches for `theme`.

    Index: `youtube_comment_topic_embedding_index` on `YouTubeCommentTopic`.
    Ranks by: best score × rel.weight per video; up to 3 matching topic names per video.
    Best for: *"show me videos where commenters discuss X"* (evidence / examples).
    Returns: video_id, title, creator, url, views, relevance, matches.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            EXAMPLE_VIDEOS_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
            platform=YOUTUBE_PLATFORM,
        )]

def comment_discussions(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Find videos whose comment-section summary is semantically similar to `theme`.

    Index: `video_summary_embedding_index` on `comment_summary_embedding`.
    Ranks by: raw vector similarity (no per-topic weights).
    Best for: *"whose comment section discusses X?"* / narrative comment themes.
    Returns: video_id, title, creator, url, score, comment_summary_description.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            COMMENT_DISCUSSIONS_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
        )]

def content_videos(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Find videos whose on-video content matches `theme` (not comment topics).

    Index: `video_content_embedding_index` (title, description, thumbnail text/keywords, tags).
    Ranks by: vector similarity on video content embedding.
    Best for: *"which videos are about / show X?"* including visual or title/description cues.
    Returns: video_id, title, creator, url, views, thumbnail fields, tags, score.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            CONTENT_VIDEOS_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
        )]

def content_top_creators(theme: str, top_n: int = 10, k: int = 2000, min_score: float = 0.0):
    """Rank creators by how much their published videos match `theme` (content signal).

    Index: `video_content_embedding_index`; aggregates content hits per channel.
    Ranks by: sum(content vector scores) across matching videos.
    Best for: *"which creators make content about X?"* (not audience comment topics).
    Returns: creator, channel_id, relevance, video_count, sample_videos.
    """
    q = embed_query(theme)
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(
            CONTENT_TOP_CREATORS_CYPHER,
            q=q,
            k=max(k, top_n),
            top_n=top_n,
            min_score=min_score,
            platform=YOUTUBE_PLATFORM,
        )]


# --- Unified fused search ---------------------------------------------------
# Runs all three video-level indexes and fuses rankings with Reciprocal Rank
# Fusion (RRF). RRF is robust against differences in score scales between
# indexes and is the standard way to combine heterogeneous retrievers.

UNIFIED_CONTENT_Q = '''
CALL db.index.vector.queryNodes('video_content_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
RETURN v.video_id AS video_id, score
'''

UNIFIED_SUMMARY_Q = '''
CALL db.index.vector.queryNodes('video_summary_embedding_index', $k, $q) YIELD node AS v, score
WHERE score >= $min_score
RETURN v.video_id AS video_id, score
'''

UNIFIED_TOPIC_Q = '''
CALL db.index.vector.queryNodes('youtube_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'youtube') = $platform
  AND score >= $min_score
MATCH (v:YouTubeVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'youtube') = $platform
WITH v.video_id AS video_id, max(score * coalesce(rel.weight, 1.0)) AS score
RETURN video_id, score
'''

UNIFIED_HYDRATE = '''
UNWIND $video_ids AS vid
MATCH (v:YouTubeVideo {video_id: vid})
OPTIONAL MATCH (c:YouTubeChannel)-[hv:HAS_VIDEO]->(v)
WHERE coalesce(hv.platform, "YouTube") = "YouTube"
RETURN v.video_id AS video_id,
       v.video_title AS title,
       coalesce(c.title, v.channel_title, v.username) AS creator,
       coalesce(c.channel_id, v.channel_id) AS channel_id,
       v.video_url AS url,
       v.view_count AS views,
       v.thumbnail_description AS thumbnail_description,
       v.video_description AS video_description,
       v.comment_summary_description AS comment_summary_description
'''


def _rrf_rank(rows, id_key: str = 'video_id', k_const: int = 60) -> Dict[str, float]:
    """Convert an ordered hit list into Reciprocal Rank Fusion scores: 1 / (k_const + rank + 1)."""
    scores: Dict[str, float] = {}
    for rank, row in enumerate(rows):
        scores[row[id_key]] = 1.0 / (k_const + rank + 1)
    return scores

def unified_search(
    theme: str,
    top_n: int = 10,
    k: int = 2000,
    min_score: float = 0.0,
    include_creator_rollup: bool = True,
):
    """Fuse video hits from content, comment-summary, and comment-topic indexes (RRF).

    Indexes: `video_content_embedding_index`, `video_summary_embedding_index`,
    `youtube_comment_topic_embedding_index`.
    Ranks by: sum of reciprocal-rank scores per video across lists (robust to score scale).
    Best for: ambiguous queries — use when you are unsure which signal fits the question.
    Returns: {videos: [...], creators: [...]} with matched_signals and fused_score per video.
    """
    q = embed_query(theme)
    k_vec = max(k, top_n)
    with driver.session(database=NEO4J_DATABASE) as s:
        content_rows = [dict(r) for r in s.run(UNIFIED_CONTENT_Q, q=q, k=k_vec, min_score=min_score)]
        summary_rows = [dict(r) for r in s.run(UNIFIED_SUMMARY_Q, q=q, k=k_vec, min_score=min_score)]
        topic_rows = [dict(r) for r in s.run(
            UNIFIED_TOPIC_Q, q=q, k=k_vec, min_score=min_score, platform=YOUTUBE_PLATFORM
        )]

        content_rank = _rrf_rank(content_rows)
        summary_rank = _rrf_rank(summary_rows)
        topic_rank   = _rrf_rank(topic_rows)

        fused: Dict[str, float] = {}
        signals: Dict[str, List[str]] = {}
        for ranks, label in [(content_rank, 'content'), (summary_rank, 'comments'), (topic_rank, 'topic')]:
            for vid, s_val in ranks.items():
                fused[vid] = fused.get(vid, 0.0) + s_val
                signals.setdefault(vid, []).append(label)

        top_ids = sorted(fused.keys(), key=lambda x: fused[x], reverse=True)[:top_n]
        if not top_ids:
            return {'videos': [], 'creators': []}

        hydrated = {r['video_id']: dict(r) for r in s.run(UNIFIED_HYDRATE, video_ids=top_ids, platform=YOUTUBE_PLATFORM)}
        videos = []
        for vid in top_ids:
            if vid not in hydrated:
                continue
            row = hydrated[vid]
            row['fused_score'] = round(fused[vid], 5)
            row['matched_signals'] = signals[vid]
            videos.append(row)

        creators = []
        if include_creator_rollup:
            rollup: Dict[str, Dict[str, Any]] = {}
            for row in videos:
                cid = row['channel_id'] or row['creator'] or 'unknown'
                agg = rollup.setdefault(cid, {
                    'channel_id': cid,
                    'creator': row['creator'],
                    'relevance': 0.0,
                    'video_count': 0,
                    'videos': [],
                })
                agg['relevance'] += row['fused_score']
                agg['video_count'] += 1
                agg['videos'].append({
                    'video_id': row['video_id'],
                    'title': row['title'],
                    'url': row.get('url'),
                })
            creators = sorted(rollup.values(), key=lambda x: x['relevance'], reverse=True)

    return {'videos': videos, 'creators': creators}


### Run test queries

Set `THEME` (and optional `TOP_N`, `MIN_SCORE`, `K_COUNT`) in the next cell, then run each retriever cell below. All retrievers use `MIN_SCORE` and `K_COUNT` from the config cell.


In [21]:
from pprint import pprint

THEME = 'gaming'
TOP_N = 10
MIN_SCORE = 0.75   # raise after similarity analysis (e.g. 0.75 for topics)
K_COUNT = 20000    # vector top-k for all retrievers

print(f'THEME: {THEME}')
print(f'TOP_N: {TOP_N}')
print(f'MIN_SCORE: {MIN_SCORE}')
print(f'K_COUNT: {K_COUNT}')


THEME: gaming
TOP_N: 10
MIN_SCORE: 0.75
K_COUNT: 20000


In [22]:
print(f'--- top_creators: comment-topic audience for "{THEME}" ---')
for row in top_creators(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- top_creators: comment-topic audience for "gaming" ---
{'channel_id': 'UCxmFU73cNUBMdmAcFtusHMg',
 'creator': 'JordyStorm',
 'relevance': 3.5186221933364874,
 'sample_topics': ['Gaming (GTA)',
                   'Gaming (Fortnite)',
                   'Gaming Skills',
                   'Gaming Performance',
                   'Gaming and Challenges'],
 'sample_videos': [{'title': 'DUO CASHCUP ROUND 1😈 | Fortnite (Nederlands)',
                    'url': 'https://www.youtube.com/watch?v=YPtVP9Kj9r4',
                    'video_id': 'YPtVP9Kj9r4'},
                   {'title': 'VERJAARDAG STREAM🎊',
                    'url': 'https://www.youtube.com/watch?v=gCP3ieeeUV8',
                    'video_id': 'gCP3ieeeUV8'},
                   {'title': 'Na 3 Maanden Ruzie, Joinde Ik Zijn Lobby!🤬',
                    'url': 'https://www.youtube.com/watch?v=E4pqSgsk_eU',
                    'video_id': 'E4pqSgsk_eU'},
                   {'title': '🔴LIVE CONTENDER CUP TOERNOOI | Fortnite '
 

In [23]:
print(f'--- top_videos_by_topic_engagement: popular videos discussing "{THEME}" ---')
for row in top_videos_by_topic_engagement(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- top_videos_by_topic_engagement: popular videos discussing "gaming" ---
{'best_topic_strength': 0.76971,
 'channel_id': 'UCilwZiBBfI9X6yiZRzWty8Q',
 'comment_count': 5332,
 'creator': 'FaZe Rug',
 'engagement_score': 6.60541,
 'log_comments': 8.58167,
 'matches': [{'score': 0.7697110176086426,
              'topic': 'Secret Gaming Room',
              'weight': 1.0}],
 'title': 'I Built a SECRET Gaming Room to Hide From My Friends!',
 'url': 'https://www.youtube.com/watch?v=GGXgnukZRMM',
 'video_id': 'GGXgnukZRMM'}
{'best_topic_strength': 0.67885,
 'channel_id': 'UCm-X6o81nRsXQTmqpyArkBQ',
 'comment_count': 6550,
 'creator': 'Flamingo',
 'engagement_score': 5.9653,
 'log_comments': 8.78737,
 'matches': [{'score': 0.7714195251464844,
              'topic': 'Gambling',
              'weight': 0.88}],
 'title': 'ROBLOX IS SCREWED',
 'url': 'https://www.youtube.com/watch?v=EfS1nhIWWbw',
 'video_id': 'EfS1nhIWWbw'}
{'best_topic_strength': 0.76896,
 'channel_id': 'UCRaehQPWXnJ72WmE8nvvKKw

In [24]:
print(f'--- top_creators_by_topic_engagement: popular creators discussing "{THEME}" ---')
for row in top_creators_by_topic_engagement(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- top_creators_by_topic_engagement: popular creators discussing "gaming" ---
{'channel_id': 'UC0DZmkupLYwc0yDsfocLh0A',
 'creator': 'Jelly',
 'engagement_score': 11.70721,
 'sample_topics': [['Gaming (Roblox, GTA, Fortnite, etc.)'],
                   ['Gameplay and Games'],
                   ['Gaming Content Suggestions'],
                   ['GTA and Gaming Content'],
                   ['Video Games (Fortnite, COD, GTA, Pokémon, Roblox, '
                    'RuneScape)']],
 'sample_videos': [{'comment_count': 270,
                    'engagement_score': 1.2925,
                    'title': 'I Crashed And Broke All My Bones…',
                    'url': 'https://www.youtube.com/watch?v=YYJlk_bX09k',
                    'video_id': 'YYJlk_bX09k'},
                   {'comment_count': 372,
                    'engagement_score': 4.65955,
                    'title': 'BOY vs. GIRLS CHALLENGE In Flee The Facility! '
                             '(Roblox)',
                    'url': 

In [25]:
print(f'--- count_creators_by_topic: how many creators discuss "{THEME}" ---')
pprint(count_creators_by_topic(THEME, min_score=MIN_SCORE, k=K_COUNT))


--- count_creators_by_topic: how many creators discuss "gaming" ---
{'creator_count': 74,
 'k': 20000,
 'min_score': 0.75,
 'sample_creators': [{'best_weighted_score': 0.0475468397140503,
                      'channel_id': 'UCWyvlIiY3RMwjFaCnpzyrDQ',
                      'creator': 'Tim Tygran',
                      'video_count': 1},
                     {'best_weighted_score': 0.24354904174804687,
                      'channel_id': 'UCpf4pGjVXwl5t8jl6zaTIiA',
                      'creator': 'Alex Klein',
                      'video_count': 11},
                     {'best_weighted_score': 0.19014196395874025,
                      'channel_id': 'UCuQBlF1YSyDI20EnqRaF5rA',
                      'creator': 'LinkTijger',
                      'video_count': 2},
                     {'best_weighted_score': 0.348902792930603,
                      'channel_id': 'UC9SeqSlo6OiBLuHjKqnu8Pg',
                      'creator': 'Games4Real',
                      'video_count': 17},
      

In [26]:
print(f'--- count_creators_by_content: how many creators publish about "{THEME}" ---')
pprint(count_creators_by_content(THEME, min_score=MIN_SCORE, k=K_COUNT))


--- count_creators_by_content: how many creators publish about "gaming" ---
{'creator_count': 1,
 'k': 20000,
 'min_score': 0.75,
 'sample_creators': [{'best_score': 0.7504673004150391,
                      'channel_id': 'UCnhZnnJOx-WnAH7HINmS8qQ',
                      'creator': 'Danny Aarons',
                      'video_count': 1}],
 'signal': 'video_content',
 'theme': 'gaming',
 'video_link_rows': 1}


In [27]:
print(f'--- count_videos_by_topic: videos with comment-topic matches for "{THEME}" ---')
pprint(count_videos_by_topic(THEME, min_score=MIN_SCORE, k=K_COUNT))


--- count_videos_by_topic: videos with comment-topic matches for "gaming" ---
{'k': 20000,
 'min_score': 0.75,
 'sample_videos': [{'best_weighted_score': 0.0475468397140503,
                    'creator': 'Tim Tygran',
                    'title': 'MIJN VERBORGEN TALENT GEVONDEN! 🎯',
                    'url': 'https://www.youtube.com/watch?v=CcaIJ1vXbtQ',
                    'video_id': 'CcaIJ1vXbtQ'},
                   {'best_weighted_score': 0.04753801822662354,
                    'creator': 'Alex Klein',
                    'title': 'Alex Is VERLIEFD in Minecraft!',
                    'url': 'https://www.youtube.com/watch?v=FN3Skk47DGk',
                    'video_id': 'FN3Skk47DGk'},
                   {'best_weighted_score': 0.19014196395874025,
                    'creator': 'LinkTijger',
                    'title': 'Don houdt niet van mijn feitjes! #fyp '
                             '#foryoupage #pokemon #151 @GameMeneer',
                    'url': 'https://www.youtube.co

In [28]:
print(f'--- count_videos_by_comment_summary: comment sections discussing "{THEME}" ---')
pprint(count_videos_by_comment_summary(THEME, min_score=MIN_SCORE, k=K_COUNT))


--- count_videos_by_comment_summary: comment sections discussing "gaming" ---
{'k': 20000,
 'min_score': 0.75,
 'sample_videos': [],
 'signal': 'comment_summary',
 'theme': 'gaming',
 'video_count': 0}


In [29]:
print(f'--- count_videos_by_content: videos about / showing "{THEME}" ---')
pprint(count_videos_by_content(THEME, min_score=MIN_SCORE, k=K_COUNT))


--- count_videos_by_content: videos about / showing "gaming" ---
{'k': 20000,
 'min_score': 0.75,
 'sample_videos': [{'creator': 'Danny Aarons',
                    'score': 0.7504673004150391,
                    'title': 'I Went To The BIGGEST Gaming Festival!',
                    'url': 'https://www.youtube.com/watch?v=cGub_BU0g0c',
                    'video_id': 'cGub_BU0g0c'}],
 'signal': 'video_content',
 'theme': 'gaming',
 'video_count': 1}


In [30]:
print(f'--- example_videos: comment-topic evidence for "{THEME}" ---')
for row in example_videos(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- example_videos: comment-topic evidence for "gaming" ---
{'creator': 'Clonny Games',
 'matches': [{'score': 0.8390703201293945,
              'topic': 'Gaming and Gameplay',
              'weight': 1.0}],
 'relevance': 0.8390703201293945,
 'title': 'COLOR OR DIE met *KIJKERS* !! | 🔴 [LIVE]',
 'url': 'https://www.youtube.com/watch?v=znZZS9YNVRM',
 'video_id': 'znZZS9YNVRM',
 'views': 14409}
{'creator': 'Jelly',
 'matches': [{'score': 0.7867941856384277,
              'topic': 'Gameplay and Games',
              'weight': 1.0}],
 'relevance': 0.7867941856384277,
 'title': 'BOY vs. GIRLS CHALLENGE In Flee The Facility! (Roblox)',
 'url': 'https://www.youtube.com/watch?v=nV2AoMTs6GU',
 'video_id': 'nV2AoMTs6GU',
 'views': 208678}
{'creator': 'FaZe Rug',
 'matches': [{'score': 0.7697005271911621,
              'topic': 'Secret Gaming Room',
              'weight': 1.0}],
 'relevance': 0.7697005271911621,
 'title': 'I Built a SECRET Gaming Room to Hide From My Friends!',
 'url': 'https://

In [32]:
print(f'--- comment_discussions: comment summaries matching "{THEME}" ---')
for row in comment_discussions(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- comment_discussions: comment summaries matching "gaming" ---


In [33]:
print(f'--- content_videos: videos about / showing "{THEME}" ---')
for row in content_videos(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- content_videos: videos about / showing "gaming" ---
{'channel_id': 'UCnhZnnJOx-WnAH7HINmS8qQ',
 'creator': 'Danny Aarons',
 'score': 0.7504673004150391,
 'tags': ['gaming convention', 'insomnia gaming festival'],
 'thumbnail_description': 'The image shows a person, likely a young adult or '
                          'teenager, wearing a green sweatshirt and holding a '
                          'white video game controller. The background is '
                          'vibrant and colorful, featuring what appears to be '
                          'a digitally-enhanced or fantasy environment '
                          'resembling an arena or stadium with large screens, '
                          'neon lighting, and crowds of people. The '
                          'composition centers the person prominently in the '
                          'foreground, while the dynamic lighting and vivid '
                          'color palette (dominated by purples, blues, and '
           

In [34]:
print(f'--- content_top_creators: creators whose videos depict "{THEME}" ---')
for row in content_top_creators(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE):
    pprint(row)


--- content_top_creators: creators whose videos depict "gaming" ---
{'channel_id': 'UCnhZnnJOx-WnAH7HINmS8qQ',
 'creator': 'Danny Aarons',
 'relevance': 0.7504673004150391,
 'sample_videos': [{'score': 0.7504673004150391,
                    'title': 'I Went To The BIGGEST Gaming Festival!',
                    'url': 'https://www.youtube.com/watch?v=cGub_BU0g0c',
                    'video_id': 'cGub_BU0g0c'}],
 'video_count': 1}


In [35]:
print(f'--- unified_search: fused videos for "{THEME}" ---')
_fused = unified_search(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE)
print('Top videos (signals that matched):')
for row in _fused['videos']:
    pprint(row)


--- unified_search: fused videos for "gaming" ---
Top videos (signals that matched):
{'channel_id': 'UCnhZnnJOx-WnAH7HINmS8qQ',
 'comment_summary_description': "The comments primarily focus on Danny's "
                                'amusing encounters with celebrities like King '
                                'Bach and Steve Aoki, as well as his frequent '
                                "use of 'salam alaikum' and gestures that "
                                'highlight multicultural interactions. Viewers '
                                "enjoy Danny's signature humor, awkwardness, "
                                'and gaming references, especially his '
                                "repeated use of 'It's a walkout.' There is a "
                                'positive response to his presence in Saudi '
                                'Arabia, with fans expressing support, '
                                'appreciation, and requests for more IRL (in '
                 

In [36]:
print(f'--- unified_search: fused creator rollup for "{THEME}" ---')
_fused = unified_search(THEME, top_n=TOP_N, k=K_COUNT, min_score=MIN_SCORE)
for row in _fused['creators']:
    pprint(row)


--- unified_search: fused creator rollup for "gaming" ---
{'channel_id': 'UC9SeqSlo6OiBLuHjKqnu8Pg',
 'creator': 'Games4Real',
 'relevance': 0.03053,
 'video_count': 2,
 'videos': [{'title': 'DIT GAAT HELEMAAL MIS...😰 #shorts',
             'url': 'https://www.youtube.com/watch?v=fTUHPWEJnEQ',
             'video_id': 'fTUHPWEJnEQ'},
            {'title': 'NIEUWE *DURE* AUTO KOPEN! | GTA5 FUTURE ROLEPLAY',
             'url': 'https://www.youtube.com/watch?v=D419IrlrLFQ',
             'video_id': 'D419IrlrLFQ'}]}
{'channel_id': 'UCnhZnnJOx-WnAH7HINmS8qQ',
 'creator': 'Danny Aarons',
 'relevance': 0.01639,
 'video_count': 1,
 'videos': [{'title': 'I Went To The BIGGEST Gaming Festival!',
             'url': 'https://www.youtube.com/watch?v=cGub_BU0g0c',
             'video_id': 'cGub_BU0g0c'}]}
{'channel_id': 'UCWyvlIiY3RMwjFaCnpzyrDQ',
 'creator': 'Tim Tygran',
 'relevance': 0.01639,
 'video_count': 1,
 'videos': [{'title': 'MIJN VERBORGEN TALENT GEVONDEN! 🎯',
             'url': 'http

## Similarity score analysis (production thresholds)

For each **query string**, run vector search against all **three** indexes (`video_content_embedding`, `comment_summary_embedding`, `topic_embedding`). For each index:

- print **query name** + index name
- show an **ASCII histogram** of cosine similarity `score` (20 bins on \[0, 1\], with spill counts outside that range)
- print up to **5 example rows** per score band: **0.90–1.00**, **0.80–0.89**, **0.70–0.79**, **0.60–0.69**, **0.50–0.59**

Tune `SIMILARITY_K_FRACTION` (default **0.1** = 10%), `SIMILARITY_K_MIN` / `SIMILARITY_K_MAX`, optional fixed override `SIMILARITY_ANALYSIS_K`, and `QUERIES_FOR_THRESHOLD_TUNING`, then run the next cell.

In [ ]:
import os
from pprint import pprint
from typing import Any, Dict, List, Optional, Tuple

SIMILARITY_K_FRACTION = float(os.getenv('SIMILARITY_K_FRACTION', '0.2'))
SIMILARITY_K_MIN = int(os.getenv('SIMILARITY_K_MIN', '50'))
SIMILARITY_K_MAX = int(os.getenv('SIMILARITY_K_MAX', '25000'))

# Optional: set to an integer string to force the same k for every index (overrides 10% sizing).
SIMILARITY_K_OVERRIDE_RAW = os.getenv('SIMILARITY_ANALYSIS_K', '').strip()

_INDEXED_NODE_COUNTS_CYPHER: Dict[str, str] = {
    'video_content_embedding_index': (
        'MATCH (v:YouTubeVideo) WHERE v.video_content_embedding IS NOT NULL RETURN count(v) AS c'
    ),
    'video_summary_embedding_index': (
        'MATCH (v:YouTubeVideo) WHERE v.comment_summary_embedding IS NOT NULL RETURN count(v) AS c'
    ),
    'youtube_comment_topic_embedding_index': (
        "MATCH (t:YouTubeCommentTopic) WHERE t.embedding IS NOT NULL AND coalesce(t.platform, 'youtube') = 'youtube' RETURN count(t) AS c"
    ),
}


def _indexed_node_counts() -> Dict[str, int]:
    out: Dict[str, int] = {}
    with driver.session(database=NEO4J_DATABASE) as s:
        for index_name, cy in _INDEXED_NODE_COUNTS_CYPHER.items():
            out[index_name] = int(s.run(cy).single()['c'])
    return out


def _k_for_index(index_name: str, counts: Dict[str, int]) -> int:
    if SIMILARITY_K_OVERRIDE_RAW:
        return max(SIMILARITY_K_MIN, int(SIMILARITY_K_OVERRIDE_RAW))
    n = counts.get(index_name, 0)
    k = int(SIMILARITY_K_FRACTION * n)
    return max(SIMILARITY_K_MIN, min(SIMILARITY_K_MAX, k))


# Edit this list for production threshold experiments
QUERIES_FOR_THRESHOLD_TUNING = [
    # 'imagery of weapons or violence',
    # 'vaping',
     'gambling',
    #  'mental health',
    # 'redbull',
    # "energy drinks",
]

CONTENT_SCORES_CYPHER = '''
CALL db.index.vector.queryNodes('video_content_embedding_index', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id, v.video_title AS title, v.video_url AS url, score
'''

SUMMARY_SCORES_CYPHER = '''
CALL db.index.vector.queryNodes('video_summary_embedding_index', $k, $q) YIELD node AS v, score
RETURN v.video_id AS video_id, v.video_title AS title, v.video_url AS url, score
'''

TOPIC_SCORES_CYPHER = '''
CALL db.index.vector.queryNodes('youtube_comment_topic_embedding_index', $k, $q) YIELD node AS t, score
WHERE coalesce(t.platform, 'youtube') = 'youtube'
MATCH (v:YouTubeVideo)-[rel:HAS_COMMENT_TOPIC]->(t)
WHERE coalesce(rel.platform, 'youtube') = 'youtube'
RETURN t.name AS topic_name,
       v.video_id AS video_id, v.video_title AS title, v.video_url AS url,
       score,
       rel.weight AS topic_weight,
       rel.position AS topic_position,
       score * coalesce(rel.weight, 1.0) AS weighted_similarity
'''

# (label, low inclusive, high exclusive) except top bucket high is inclusive via next bucket start
SCORE_BANDS: List[Tuple[str, float, float]] = [
    ('0.90–1.00', 0.9, 1.0000001),
    ('0.80–0.89', 0.8, 0.9),
    ('0.70–0.79', 0.7, 0.8),
    ('0.60–0.69', 0.6, 0.7),
    ('0.50–0.59', 0.5, 0.6),
]


def _fetch_rows(cypher: str, q: List[float], k: int) -> List[Dict[str, Any]]:
    with driver.session(database=NEO4J_DATABASE) as s:
        return [dict(r) for r in s.run(cypher, q=q, k=k)]


def _scores(rows: List[Dict[str, Any]]) -> List[float]:
    return [float(r['score']) for r in rows if r.get('score') is not None]


def _ascii_histogram(scores: List[float], n_bins: int = 20, width: int = 50) -> None:
    if not scores:
        print('  (no scores)')
        return
    lo, hi = min(scores), max(scores)
    # Bin into [0,1] plus spill for outliers
    lo_b, hi_b = 0.0, 1.0
    counts = [0] * n_bins
    under = over = 0
    for s in scores:
        if s < lo_b:
            under += 1
        elif s > hi_b:
            over += 1
        else:
            idx = int((s - lo_b) / (hi_b - lo_b) * n_bins)
            if idx >= n_bins:
                idx = n_bins - 1
            counts[idx] += 1
    mx = max(counts) if counts else 1
    print(f'  score min={lo:.4f} max={hi:.4f}  (binning [0,1] with {n_bins} bins; below0={under} above1={over})')
    for i, c in enumerate(counts):
        left = lo_b + (hi_b - lo_b) * i / n_bins
        right = lo_b + (hi_b - lo_b) * (i + 1) / n_bins
        bar = '#' * max(1, int(width * c / mx)) if c else ''
        print(f'  [{left:.2f},{right:.2f}): {c:4d}  {bar}')


def _band_examples(rows: List[Dict[str, Any]], label: str, lo: float, hi: float, n: int = 5) -> None:
    band = [r for r in rows if r.get('score') is not None and lo <= float(r['score']) < hi]
    band.sort(key=lambda r: float(r['score']), reverse=True)
    print(f'  band {label}: {len(band)} hits (showing top {min(n, len(band))})')
    for r in band[:n]:
        pprint(r)


def print_topic_retriever_topic_name_matches(
    rows: List[Dict[str, Any]],
    topic_name_substring: Optional[str] = None,
) -> None:
    """Print topic retriever rows whose topic_name contains topic_name_substring (case-insensitive).

    Does not call the embedding model or Neo4j — pass rows from an existing TOPIC_SCORES_CYPHER fetch.
    If topic_name_substring is None or blank, prints nothing.
    """
    if not topic_name_substring or not str(topic_name_substring).strip():
        return
    needle = str(topic_name_substring).strip().lower()
    matched = [r for r in rows if needle in (r.get('topic_name') or '').lower()]
    if not matched:
        return
    matched.sort(
        key=lambda r: float(
            r.get('weighted_similarity')
            if r.get('weighted_similarity') is not None
            else r.get('score') or 0.0
        ),
        reverse=True,
    )
    print('-' * 88)
    print(f'  TOPIC ROWS WITH topic_name containing {topic_name_substring.strip()!r}  (n={len(matched)})')
    for r in matched:
        pprint(r)


def analyze_similarity_for_query(
    query_name: str,
    counts: Dict[str, int],
    *,
    optional_topic_name_substring: Optional[str] = None,
) -> None:
    print('=' * 88)
    print(
        f'QUERY: {query_name!r}   (k ≈ {SIMILARITY_K_FRACTION * 100:.0f}% of indexed nodes per index, '
        f'min={SIMILARITY_K_MIN} max={SIMILARITY_K_MAX}'
        + (f", override={SIMILARITY_K_OVERRIDE_RAW!r}" if SIMILARITY_K_OVERRIDE_RAW else '')
        + ')'
    )
    print('=' * 88)
    q = embed_query(query_name)

    suites: List[Tuple[str, str]] = [
        ('video_content_embedding_index', CONTENT_SCORES_CYPHER),
        ('video_summary_embedding_index', SUMMARY_SCORES_CYPHER),
        ('youtube_comment_topic_embedding_index', TOPIC_SCORES_CYPHER),
    ]

    for index_name, cypher in suites:
        k = _k_for_index(index_name, counts)
        rows = _fetch_rows(cypher, q, k)
        scores = _scores(rows)
        print('-' * 88)
        n_idx = counts.get(index_name, 0)
        print(f'INDEX: {index_name}   indexed_nodes={n_idx}   k_used={k}   rows_returned={len(rows)}')
        _ascii_histogram(scores)
        for band_label, lo, hi in SCORE_BANDS:
            _band_examples(rows, band_label, lo, hi, n=5)
        if index_name == 'youtube_comment_topic_embedding_index' and optional_topic_name_substring:
            print_topic_retriever_topic_name_matches(rows, optional_topic_name_substring)
        print()


_COUNTS_GLOBAL = _indexed_node_counts()
print('Indexed node counts (used to size k per index):', _COUNTS_GLOBAL)

for _q in QUERIES_FOR_THRESHOLD_TUNING:
    analyze_similarity_for_query(_q, counts=_COUNTS_GLOBAL)
